### 1.3.5.11. Critical Points and the Second Derivative Test

$$
\nabla f(\mathbf{x}_0) = \mathbf{0}; \qquad
D = f_{xx}f_{yy} - (f_{xy})^{2} = \det H .
$$

$$
D > 0,\ f_{xx} > 0 \Rightarrow \text{local min}; \quad
D > 0,\ f_{xx} < 0 \Rightarrow \text{local max}; \quad
D < 0 \Rightarrow \text{saddle}.
$$

**Explanation:**

A critical point is where the [gradient](./03_gradient.ipynb) vanishes — the first-order necessary condition for an interior extremum. To classify it, the second derivative test inspects the [Hessian](./09_hessian_matrix.ipynb): in two variables the discriminant $D = f_{xx}f_{yy} - f_{xy}^2$ is exactly $\det H$, and its sign together with $f_{xx}$ distinguishes minima, maxima, and saddle points. In general dimensions the test becomes Hessian [definiteness](../../01_Linear_Algebra/07_Theoretical_Linear_Algebra/18_matrix_definiteness.ipynb) — positive definite for a minimum, indefinite for a saddle — which is the [sufficient second-order optimality condition](../../../03_Optimization/01_Optimization_Fundamentals/05_convexity_and_sufficient_conditions.ipynb) of unconstrained optimization.

**Properties:**
- $D < 0$ means $H$ is indefinite (eigenvalues of opposite sign): a saddle point.
- $D = 0$ is inconclusive; higher-order information is needed.

**Intuition:**

The second-derivative test distinguishes a local minimum (a), a local maximum (b), and a saddle point (c) by the curvature of the surface.

<p align="center">
  <img src="../../../Figures/01030511_second_derivative_test_classification.jpeg"
       alt="Surfaces with a local minimum, a local maximum, and a saddle point"
       width="560">
</p>

**Numerical Example:**

Let $f(x,y) = x^3 + y^3 - 3xy$. Setting $\nabla f = \mathbf{0}$: $f_x = 3x^2 - 3y = 0$ and $f_y = 3y^2 - 3x = 0$ give $y = x^2$ and $x = y^2$, hence the critical points $(0,0)$ and $(1,1)$. With $f_{xx} = 6x$, $f_{yy} = 6y$, $f_{xy} = -3$:

$$
(0,0):\ D = (0)(0) - (-3)^2 = -9 < 0 \Rightarrow \textbf{saddle},
$$

$$
(1,1):\ D = (6)(6) - (-3)^2 = 27 > 0,\ f_{xx} = 6 > 0 \Rightarrow \textbf{local minimum},\ f(1,1) = -1 .
$$

In [1]:
import sympy as sp

x, y = sp.symbols("x y")
f = x**3 + y**3 - 3 * x * y
variables = [x, y]

gradient = [sp.diff(f, variable) for variable in variables]
critical_points = sp.solve(gradient, variables, dict=True)
hessian = sp.hessian(f, variables)

for point in critical_points:
    if all(value.is_real for value in point.values()):
        discriminant = hessian.subs(point).det()
        curvature = hessian.subs(point)[0, 0]
        if discriminant < 0:
            label = "saddle"
        elif curvature > 0:
            label = "local min"
        else:
            label = "local max"
        print(f"critical point {(point[x], point[y])}: D = {discriminant}, f_xx = {curvature} -> {label}")

critical point (0, 0): D = -9, f_xx = 0 -> saddle
critical point (1, 1): D = 27, f_xx = 6 -> local min


**Equivalent CasADi implementation:**

CasADi finds the same critical points with a root-finder on the gradient and classifies each by the determinant of the Hessian.

In [2]:
import casadi as ca

variables = ca.SX.sym("x", 2)
f = variables[0]**3 + variables[1]**3 - 3 * variables[0] * variables[1]
gradient = ca.gradient(f, variables)
hessian, _ = ca.hessian(f, variables)

root_finder = ca.rootfinder("r", "newton", {"x": variables, "g": gradient})
hessian_function = ca.Function("H", [variables], [hessian])

for start in (ca.DM([0.2, 0.2]), ca.DM([1.2, 1.2])):
    critical_point = root_finder(x0=start)["x"]
    discriminant = float(ca.det(hessian_function(critical_point)))
    label = "saddle" if discriminant < 0 else "local min/max"
    print("critical point", [round(float(value), 4) for value in critical_point.full().ravel()],
          "det H =", round(discriminant, 4), "->", label)

critical point [-0.0, -0.0] det H = -9.0 -> saddle
critical point [1.0, 1.0] det H = 27.0 -> local min/max


**References:**

[📘 Strang, G. (2016). *Calculus Volume 3*. OpenStax.](https://openstax.org/details/books/calculus-volume-3)  
[📗 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*, Appendix A.4.](https://web.stanford.edu/~boyd/cvxbook/)

---

[⬅️ Previous: Second-Order Taylor Expansion](./10_second_order_taylor_expansion.ipynb) | [Next: Implicit Function Theorem ➡️](./12_implicit_function_theorem.ipynb)